In [ ]:
from votekit.cvr_loaders import load_scottish
from votekit.elections import STV, fractional_transfer

In [ ]:
b_bloc_parties = ['Scottish National Party (SNP)', 'Green (Gr)']

In [ ]:
file_names = {("fife", 2022, 21): "fife_2022_ward21.csv",
              ("aberdeen", 2017, 12) : "aberdeen_2017_ward12.csv",
              ("aberdeen", 2022, 12): "aberdeen_2022_ward12.csv",
              ("angus", 2012, 8): "angus_2012_ward8.csv",
              ("falkirk", 2017, 6): "falkirk_2017_ward6.csv",
              ("clackmannanshire", 2012, 2): "clackmannanshire_2012_ward2.csv",
              ("renfrewshire", 2017, 1): "renfrewshire_2017_ward1.csv",
              ("glasgow", 2012, 16): "glasgow_2012_ward16.csv",
              ("north ayrshire", 2022, 1): "north_ayrshire_2022_north_coast.csv"
              }

In [ ]:
# the load_blt function returns a tuple, the first element is the preference
# profile and the second is the number of seats in the election
for key, file_name in file_names.items():
    city, year, ward = key
    print(f"{city} ward_{ward} {year}")

    scottish_profile, num_seats, cand_list, cand_to_party, ward = load_scottish(file_name)
    cand_to_bloc = {c:"B" if cand_to_party[c] in b_bloc_parties 
                else "A" for c in cand_list}
    num_voters = scottish_profile.num_ballots()

    # record first place votes for B as our parameter and as our measure of proportion of population
    pi_b = sum([b.weight for b in scottish_profile.ballots  if cand_to_bloc[list(b.ranking[0])[0]] == "B"]) / float(num_voters)
    print(f"B bloc share {pi_b:.2f}")

    prop_target = pi_b
    print(f"Proportionality target {prop_target*num_seats:.2f}")

    # record number of B winners
    election  =STV(profile = scottish_profile, transfer=fractional_transfer, seats = num_seats)
    results = election.run_election()
    winners= [c for s in results.winners() for c in s]
    num_B_winners = len([c for c in winners if cand_to_bloc[c] == "B"])

    print("Seats won by B bloc", num_B_winners)
    print()

In [ ]:
for key, file_name in file_names.items():
    city, year, ward = key
    print(f"{city} ward_{ward} {year}")

    scottish_profile, num_seats, cand_list, cand_to_party, ward = load_scottish(file_name)
    cand_to_bloc = {c:"B" if cand_to_party[c] in b_bloc_parties 
                else "A" for c in cand_list}
    
    num_cands = len(cand_to_bloc)
    borda_vector = [num_seats-i for i in range(num_seats)] + [0]*(num_cands-num_seats)
    
    print(borda_vector)
    b_borda = 0.0
    total_borda = 0.0
    for ballot in scottish_profile.ballots:
        b_indices = []
        for i,s in enumerate(ballot.ranking):
            if len(s)>1:
                raise ValueError("we do not handle ties in ballots")
            
            candidate, = s
            if cand_to_bloc[candidate] == "B":
                b_indices.append(i)
        
        total_borda+= int(ballot.weight)*sum(borda_vector[:len(ballot.ranking)])
        b_borda += int(ballot.weight)*sum([s for i,s in enumerate(borda_vector) if i in b_indices])
    
    borda_share = b_borda/total_borda
    print(f"B bloc borda share {borda_share:.2f}")
    print(f"Proportionality target {borda_share * num_seats:.2f}")


    # record number of B winners
    election  =STV(profile = scottish_profile, transfer=fractional_transfer, seats = num_seats)
    results = election.run_election()
    winners= [c for s in results.winners() for c in s]
    num_B_winners = len([c for c in winners if cand_to_bloc[c] == "B"])

    print("Seats won by B bloc", num_B_winners)
    print()
    